# LangChain Demystified: Building LLM-Powered Applications from Scratch

A hands-on technical notebook covering LangChain's core components and real-world use cases using Google Gemini as the underlying language model.

**Author:** Sushil Panthi  
**Internship:** GenAI Intern, Innomatics Research Labs  
**Stack:** Python, LangChain, Google Gemini API, FAISS

---

### What this notebook covers

This notebook is the working code companion to the technical blog on LangChain. Each section maps directly to a concept explained in the blog, with runnable implementations that you can test and modify.

The flow follows: LLMs and Chat Models → Prompt Templates → Chains → Memory → Tools → Agents → Document Loaders → Vector Stores → three complete real-world use cases.

---

### Table of Contents

- Section 0 — Setup and Installation
- Section 1 — LLMs and Chat Models
- Section 2 — Prompt Templates
- Section 3 — Chains using LCEL
- Section 4 — Memory and Conversation History
- Section 5 — Tools
- Section 6 — Agents
- Section 7 — Document Loaders
- Section 8 — Vector Stores with FAISS
- Use Case 1 — RAG Document Q&A
- Use Case 2 — Autonomous Research Agent
- Use Case 3 — Customer Support Bot with Memory

---
## Section 0: Setup and Installation

Run this cell first to install all required packages. Once it finishes, go to **Runtime → Restart Runtime** before running any other cell. This ensures all packages are properly loaded into the environment.

In [1]:
# Install all dependencies needed for this notebook
# Using Google Gemini as the LLM provider (free tier available)

!pip install -q --upgrade --force-reinstall --no-cache-dir \
               langchain langchain-google-genai langchain-community \
               langchain-core langchain-text-splitters \
               faiss-cpu wikipedia tiktoken pypdf

print("Installation complete.")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 155.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 140.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 229.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 247.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 138.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 166.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 266.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 187.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 311.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 185.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 353.8 MB/s eta 0:00

In [2]:
# Verify all packages are correctly installed after runtime restart

import langchain
import langchain_google_genai
import langchain_community
import importlib.metadata

print("Package versions:")
print(f"  langchain              : {langchain.__version__}")
print(f"  langchain-google-genai : {importlib.metadata.version('langchain-google-genai')}")
print(f"  langchain-community    : {importlib.metadata.version('langchain-community')}")
print("\nEnvironment is ready.")

Package versions:
  langchain              : 1.2.15
  langchain-google-genai : 4.2.1
  langchain-community    : 0.4.1

Environment is ready.


In [5]:
# Set your Google Gemini API key
# Get a free key from: https://aistudio.google.com → Get API Key
# The key will not be visible in output after you type it

import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
print("API key saved for this session.")

Enter your Gemini API key: ··········
API key saved for this session.


---
## Section 1: LLMs and Chat Models

The first thing to understand is the distinction between an LLM and a Chat Model. A plain LLM takes a raw string and returns a raw string. A Chat Model, on the other hand, accepts a structured list of messages, each labeled with a role: system, human, or AI. In production applications, the chat model format is almost always preferable because it lets you separate the system-level instructions from the actual user input clearly.

LangChain abstracts the provider completely. Switching from Gemini to another model later is a one-line change in the initialization. Everything else in your pipeline remains untouched.

In [6]:
# Basic LLM call using Google Gemini through LangChain
# ChatGoogleGenerativeAI wraps Gemini in the standard LangChain chat interface

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize the Gemini model
# If you encounter 404 NOT_FOUND errors, it means the model is not accessible.
# Refer to the ListModels cell below to find an available model.
llm = ChatGoogleGenerativeAI(
    model="models/gemini-flash-latest", # Updated model based on available models list
    temperature=0.7,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Structured message format: system sets the context, human sends the query
messages = [
    SystemMessage(content="You are a helpful data science assistant."),
    HumanMessage(content="Explain overfitting in one sentence.")
]

response = llm.invoke(messages)
print("Model response:")
print(response.content)

Model response:
[{'type': 'text', 'text': 'Overfitting occurs when a machine learning model learns the noise and specific details of the training data so thoroughly that it fails to generalize accurately to new, unseen data.', 'extras': {'signature': 'Ev8HCvwHAb4+9vv5LvlyC/UuQ9aK0z/dKLL5s0o60nav4SsF55VXIqWzs/dmg+InN+IqJCI52abuWP3pmg3mTEHGw4esYwotlT94F1ppdmLCM/FXdxR5vqeNdW9aw2BAp0FvothLTAYsEmHqrSFYVHDIDx0kx5FhW4AV9lhmk1dD+a+kXECRyHqs9eiTWqnLQwYY5KnYJDTmr0b3r4j8e2PweYCdw0qOOiSCHZOpZDd3RTqALKQWCaiHNiOV+NNZRsXvGDm/Gh6x6fu+32/JbKK/crIguiwIrTbN1UWQOxukAh861WGiZxt/9kHObiz2d88LSpyO34q+tywwWm9C2MDIqhup4uecTtBmEr3x+vvH8k8+/ECVv/5x4pp2xXqE5nuu+SyKadsaKXV1NhoinNYpXNYHPD6l4/nvzahJ5HaJlvvO3qPjHHpR5sUVppIvKsBiZkCOMs1/T1trev7x8YQcK8qXPxNd6gJPR7plEcMcKYEUuBtyqiA1NvD5AqflKyoMIbGbaM1iRugUV+nSaXy8IuKgjKr0F0xt2m+PWgASwFngI4GtS0OCZABkbtSZpaYouQkUnkaTPQpwFYLkyntqddkyFihdU/o8fjR5pdAcOkuWNq16tfSeuF1B45vRnIOx3S2o4BSUdRydQo5uB7vLsohmPv+7Lns10w5KHC3WzwQgepzxRFYVYkqUbPeLOUPVgSSA5CMueuCLgS0JkZl/USGSmZkIilY4cD3JndMz

In [7]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

print("Available Gemini models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(f"  - {m.name} (description: {m.description})")
print("\nPlease pick one of the available model names and update the `model` parameter in the `llm` initialization cell (BofKWKqe00yW).")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Gemini models:
  - models/gemini-2.5-flash (description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.)
  - models/gemini-2.5-pro (description: Stable release (June 17th, 2025) of Gemini 2.5 Pro)
  - models/gemini-2.0-flash (description: Gemini 2.0 Flash)
  - models/gemini-2.0-flash-001 (description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025.)
  - models/gemini-2.0-flash-lite-001 (description: Stable version of Gemini 2.0 Flash-Lite)
  - models/gemini-2.0-flash-lite (description: Gemini 2.0 Flash-Lite)
  - models/gemini-2.5-flash-preview-tts (description: Gemini 2.5 Flash Preview TTS)
  - models/gemini-2.5-pro-preview-tts (description: Gemini 2.5 Pro Preview TTS)
  - models/gemma-3-1b-it (description: )
  - models/gemma-3-4b-it (description: )
  - models/gemma-3-12b-it (description: )
  - models/gem

## Diagnostic: List Available Models

If you are encountering `404 NOT_FOUND` errors with `gemini-1.5-flash` or `gemini-pro`, it means these models are not accessible with your current API key or in your region. Run the following cell to list all models that are available to you. You can then update the `model` parameter in the `llm` initialization to one of the models listed.

In [8]:
# Prompt Templates: reusable prompt structures with dynamic variables

from langchain_core.prompts import ChatPromptTemplate

# Define a template with two variables: domain and question
# These act like function parameters for your prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Keep your answers concise and precise."),
    ("human", "{question}")
])

# Fill in the template with actual values
formatted = prompt.invoke({
    "domain": "machine learning",
    "question": "What is the vanishing gradient problem and why does it happen?"
})

# The formatted output is a list of messages ready to send to the model
response = llm.invoke(formatted)
print("Response:")
print(response.content)

# Show what the formatted messages actually look like under the hood
print("\nFormatted messages sent to the model:")
for msg in formatted.messages:
    print(f"  [{msg.__class__.__name__}]: {msg.content}")

Response:
[{'type': 'text', 'text': 'The **vanishing gradient problem** occurs during the training of deep neural networks when the gradients used to update the weights become extremely small as they are backpropagated from the output layer to the early layers. Consequently, the weights in the initial layers change very little, preventing the network from learning effectively.\n\n### Why it happens:\n1.  **Chain Rule Multiplications:** Backpropagation calculates gradients by multiplying the local derivatives of each layer. In a deep network, the gradient at an early layer is the product of many terms.\n2.  **Saturating Activation Functions:** Functions like **Sigmoid** or **Tanh** have derivatives that are less than 1.0 (Sigmoid’s maximum derivative is 0.25). \n3.  **Exponential Decay:** When you multiply many values between 0 and 1 together across dozens of layers, the product shrinks exponentially toward zero.\n\n**Result:** The "signal" from the error at the output is lost by the ti

---
## Section 3: Chains using LCEL

LangChain Expression Language (LCEL) is the modern way to compose components. Instead of wrapping things in objects and nesting function calls, LCEL uses the pipe operator to connect steps in a readable left-to-right flow.

The pattern is always: prompt → model → output parser. Each component takes input, transforms it, and passes the result to the next. This is similar to Unix pipes, where the output of one command becomes the input of the next.

This approach also makes streaming straightforward, which is important when you want to display responses token by token in a user interface.

In [9]:
# Building a chain with LCEL — the pipe operator connects components

from langchain_core.output_parsers import StrOutputParser

# Three components connected with | (pipe)
# Data flows left to right: prompt → llm → parser
chain = prompt | llm | StrOutputParser()

# Invoke the entire pipeline with a single dictionary
result = chain.invoke({
    "domain": "deep learning",
    "question": "Why does batch normalization improve training stability?"
})

print("Chain output (plain string after parsing):")
print(result)
print(f"\nOutput type: {type(result)}")

Chain output (plain string after parsing):
Batch Normalization (BN) improves training stability through several key mechanisms:

1.  **Smoother Loss Landscape:** BN reparameterizes the underlying optimization problem, making the loss surface significantly smoother (more Lipschitz-continuous). This results in more predictable and stable gradients.
2.  **Reduced Gradient Dependence on Scale:** It decouples the magnitude of weights from the magnitude of activations. This prevents small changes in early layers from amplifying into massive swings in deeper layers, mitigating exploding or vanishing gradients.
3.  **Higher Learning Rates:** Because the loss landscape is smoother and gradients are more stable, models can be trained with much higher learning rates without risking divergence, leading to faster and more robust convergence.
4.  **Activation Regulation:** By keeping inputs to activation functions (like ReLU or Sigmoid) within a controlled range (typically zero mean and unit varianc

In [10]:
# Streaming with LCEL — tokens arrive one at a time
# This is how you would display real-time output in a chatbot UI

print("Streaming response (tokens arrive incrementally):\n")

for chunk in chain.stream({
    "domain": "data science",
    "question": "What is the difference between bagging and boosting?"
}):
    print(chunk, end="", flush=True)

print("\n\nStreaming complete.")

Streaming response (tokens arrive incrementally):

The primary differences between **Bagging** (Bootstrap Aggregating) and **Boosting** are:

### 1. Training Process
*   **Bagging:** Parallel. Multiple independent models are trained simultaneously on different subsets of the data.
*   **Boosting:** Sequential. Models are trained one after another, where each new model attempts to correct the errors made by the previous ones.

### 2. Data Selection
*   **Bagging:** Uses **Bootstrap sampling** (random sampling with replacement). Each model gets an independent random subset.
*   **Boosting:** Uses **Weighted sampling**. Observations that were misclassified by previous models are assigned higher weights so the next model focuses more on them.

### 3. Objective
*   **Bagging:** Primarily reduces **variance** (prevents overfitting). It smooths out the predictions of complex, high-variance models.
*   **Boosting:** Primarily reduces **bias** (improves underfitting). It combines multiple "weak

---
## Section 4: Memory and Conversation History

Every call to a language model is stateless by default. The model has no knowledge of what was said in a previous turn. This is fine for single-shot queries but breaks down immediately for anything conversational.

LangChain's memory system solves this by maintaining a history of messages per session. When a new message arrives, the history is injected into the prompt automatically before the model sees it. The model then has full context of the conversation without any manual tracking on your part.

The key design here is the session ID. Each conversation thread gets a unique identifier, so multiple users or multiple conversations can run in parallel without interfering with each other.

In [12]:
# Memory: maintaining conversation state across multiple turns
# Uses RunnableWithMessageHistory — the current recommended approach

from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# A simple dictionary acts as our session store
# In production this would be a database like Redis
store = {}

def get_session_history(session_id: str):
    """Retrieve or create a message history for the given session."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# The placeholder is where the message history gets injected automatically
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Remember everything the user tells you."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}")
])

memory_chain = memory_prompt | llm | StrOutputParser()

# Wrap the chain with the memory manager
chain_with_memory = RunnableWithMessageHistory(
    memory_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Both turns use the same session_id so history is shared
session_config = {"configurable": {"session_id": "session_001"}}

turn1 = chain_with_memory.invoke(
    {"input": "My name is Aryan and I am studying machine learning."},
    config=session_config
)
print("Turn 1:")
print(turn1)

turn2 = chain_with_memory.invoke(
    {"input": "What is my name and what am I studying?"},
    config=session_config
)
print("\nTurn 2 (tests if memory is working):")
print(turn2)

ChatGoogleGenerativeAIError: Error calling model 'models/gemini-flash-latest' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 36.834678421s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMod

---
## Section 5: Tools

Language models have a knowledge cutoff. They cannot browse the web, run calculations on live data, or call external APIs on their own. Tools are the solution to this limitation.

In LangChain, a Tool is simply a Python function decorated with `@tool`. The function's docstring becomes the tool's description, and this is what the language model reads to understand when and how to use the tool. Writing a clear, accurate docstring is therefore critical — the model's ability to use the tool correctly depends entirely on it.

Tools can be custom functions you write, or pre-built integrations from the LangChain community library.

In [ ]:
# Custom tool using the @tool decorator
# The docstring is what the LLM reads to understand how to use this tool

from langchain_core.tools import tool

@tool
def calculate_compound_interest(principal: float, rate: float, years: int) -> float:
    """
    Calculate the final amount after compound interest.
    Use this when the user asks about investment growth or compound interest.

    Args:
        principal: The initial investment amount in any currency
        rate: Annual interest rate as a decimal (example: 0.08 for 8 percent)
        years: Number of years the amount is invested
    Returns:
        The total amount after compound interest is applied
    """
    return principal * (1 + rate) ** years

# Test the tool directly before handing it to an agent
test_result = calculate_compound_interest.invoke({
    "principal": 50000,
    "rate": 0.08,
    "years": 10
})

print(f"Tool output: {test_result:.2f}")
print(f"Tool name: {calculate_compound_interest.name}")
print(f"Tool description (first 100 chars): {calculate_compound_interest.description[:100]}")

In [ ]:
# Pre-built Wikipedia tool from the LangChain community package
# This is a ready-made integration that wraps the Wikipedia API

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=1,
        doc_content_chars_max=600
    )
)

# Test it directly
wiki_result = wikipedia_tool.run("LangChain AI framework")
print("Wikipedia tool output:")
print(wiki_result)

---
## Section 6: Agents

A chain is linear. You define the steps, and data flows through them in that exact order every time. An agent is different. It uses the language model itself to decide what to do next.

The agent receives a task, considers which tools are available, picks one if needed, observes the result, and then decides whether it has enough information to answer or whether it needs to take another action. This reasoning loop continues until the agent reaches a final answer.

This is what makes agents suited for open-ended tasks where you cannot predict in advance exactly what steps will be needed.

In [ ]:
# Agent that autonomously decides when and how to use tools

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent # Corrected import path for AgentExecutor
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

@tool
def calculate_compound_interest(principal: float, rate: float, years: int) -> float:
    """Calculate compound interest. Use when user asks about investment growth.
    Args: principal (initial amount), rate (annual rate as decimal), years (investment period)
    Returns: final amount after compound interest"""
    return principal * (1 + rate) ** years

# Temperature 0 for agents — we want deterministic reasoning, not creativity
agent_llm = ChatGoogleGenerativeAI(
    model="models/gemini-flash-latest", # Updated model name to a working one
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

tools = [calculate_compound_interest]

# The agent_scratchpad placeholder is where the agent writes its reasoning steps
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful financial assistant. Use tools when calculations are needed."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

agent = create_tool_calling_agent(agent_llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Running agent...\n")
result = agent_executor.invoke({
    "input": "If I invest 50000 rupees at 8 percent annual interest for 10 years, what will be my final amount?"
})

print("\nFinal answer:")
print(result["output"])

---
## Section 7: Document Loaders

Before a language model can answer questions about your private data, that data needs to be loaded and converted into a usable format. Document Loaders handle this. They standardize the ingestion process across different source types — PDFs, web pages, SQL databases, GitHub repositories — so the rest of your pipeline does not need to care about where the data came from.

After loading, documents are typically split into smaller chunks. This is necessary because language models have a context window limit, and because smaller chunks make semantic retrieval more precise. The RecursiveCharacterTextSplitter is the go-to choice because it tries to split on natural boundaries like paragraphs and sentences before falling back to character-level splitting.

In [ ]:
# Document loading and splitting
# Using a web page as the document source (no file upload needed)

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load a Wikipedia page as our document
loader = WebBaseLoader("https://en.wikipedia.org/wiki/LangChain")
documents = loader.load()

print(f"Documents loaded: {len(documents)}")
print(f"Total characters: {len(documents[0].page_content)}")
print(f"\nFirst 400 characters of raw content:")
print(documents[0].page_content[:400])

# Split into chunks
# chunk_size controls max characters per chunk
# chunk_overlap ensures context is not lost at chunk boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)
chunks = splitter.split_documents(documents)

print(f"\nTotal chunks created: {len(chunks)}")
print(f"Sample chunk content:\n{chunks[0].page_content[:300]}")

---
## Section 8: Vector Stores with FAISS

A traditional database finds records by matching exact keywords. A vector store works differently. It converts text into a high-dimensional numerical vector — an embedding — that captures the semantic meaning of the content. Two pieces of text that mean the same thing will have embeddings that are mathematically close, even if they use completely different words.

FAISS is an open-source library from Meta that enables efficient similarity search over these vectors. It runs locally, which means no additional API calls or costs for retrieval. Only the initial embedding creation requires an API call.

In [ ]:
# Vector store creation and semantic search using FAISS
# Embeddings convert text chunks into numerical vectors

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

print("Creating embeddings for all chunks...")

# Google's embedding model converts text into 768-dimensional vectors
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Store all chunk embeddings in a local FAISS index
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"Vector store created with {len(chunks)} vectors.")

# Create a retriever that returns the top 3 most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Run a semantic search — finds meaning, not just keywords
query = "What problem does LangChain solve for developers?"
results = retriever.invoke(query)

print(f"\nQuery: {query}")
print(f"Retrieved {len(results)} relevant chunks:\n")
for i, doc in enumerate(results):
    print(f"Chunk {i+1}:")
    print(doc.page_content[:250])
    print()

---
## Use Case 1: RAG-based Document Question Answering

**Problem:** A company has hundreds of internal documents. Employees spend significant time manually searching through them for specific information. A keyword search is unreliable because the same concept can be expressed in many ways.

**Solution:** A Retrieval-Augmented Generation pipeline that semantically searches the document collection, retrieves the most relevant passages, and uses the language model to generate a precise answer grounded in those passages rather than in the model's training data.

**Components used:** WebBaseLoader, RecursiveCharacterTextSplitter, GoogleGenerativeAIEmbeddings, FAISS, ChatGoogleGenerativeAI, LCEL chain

In [ ]:
# Use Case 1: Full RAG pipeline from document ingestion to answer generation

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Step 1: Load the source document
print("Step 1: Loading document...")
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Retrieval-augmented_generation")
raw_docs = loader.load()
print(f"  Loaded {len(raw_docs)} page(s), {len(raw_docs[0].page_content)} characters total.")

# Step 2: Split into chunks
print("Step 2: Splitting into chunks...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
doc_chunks = text_splitter.split_documents(raw_docs)
print(f"  Created {len(doc_chunks)} chunks.")

# Step 3: Embed and index
print("Step 3: Building vector index...")
embed_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
doc_vectorstore = FAISS.from_documents(doc_chunks, embed_model)
doc_retriever = doc_vectorstore.as_retriever(search_kwargs={"k": 4})
print("  Vector index ready.")

# Step 4: Define the RAG prompt
# The model is instructed to use only the provided context
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """Answer the question using only the context provided below.
If the answer cannot be found in the context, say so clearly rather than guessing.

Context:
{context}"""),
    ("human", "{question}")
])

def format_retrieved_docs(docs):
    """Combine retrieved chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# Step 5: Assemble the RAG chain using LCEL
# The retriever and question run in parallel, then merge into the prompt
rag_chain = (
    {"context": doc_retriever | format_retrieved_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | rag_llm
    | StrOutputParser()
)

# Step 6: Ask questions
print("\nQuerying the RAG pipeline:\n")
test_questions = [
    "What is retrieval-augmented generation and what problem does it address?",
    "What are the main components involved in a RAG system?"
]

for question in test_questions:
    print(f"Q: {question}")
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    print()

---
## Use Case 2: Autonomous Research Agent

**Problem:** Analysts performing research need to search for information, run calculations on retrieved data, and compile summaries — a workflow that currently happens across multiple browser tabs and tools.

**Solution:** An autonomous agent equipped with a Wikipedia search tool and a statistics calculator. The agent decides which tool to use based on what the task requires, processes the results, and delivers a consolidated response.

**Components used:** AgentExecutor, create_tool_calling_agent, WikipediaQueryRun, custom tool, ChatPromptTemplate

In [ ]:
# Use Case 2: Autonomous agent that combines search and calculation

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
import math

# Tool 1: Wikipedia for live information retrieval
search_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=800)
)

# Tool 2: Custom statistics calculator
@tool
def calculate_statistics(numbers: str) -> str:
    """
    Calculate descriptive statistics for a comma-separated list of numbers.
    Use this when the user provides numerical data and wants statistical analysis.
    Input format: numbers separated by commas, example: '10, 20, 30, 40'
    Returns mean, minimum, maximum, and standard deviation.
    """
    values = [float(x.strip()) for x in numbers.split(",")]
    mean_val = sum(values) / len(values)
    std_val = math.sqrt(sum((x - mean_val) ** 2 for x in values) / len(values))
    return (
        f"Count: {len(values)} | "
        f"Mean: {mean_val:.2f} | "
        f"Min: {min(values):.2f} | "
        f"Max: {max(values):.2f} | "
        f"Std Dev: {std_val:.2f}"
    )

# Build the research agent
research_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

research_tools = [search_tool, calculate_statistics]

research_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a research assistant with access to Wikipedia and a statistics calculator.
Think through the task carefully. Use tools when needed.
Always mention which tool you used and why.
End with a brief, clear summary."""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

research_agent = create_tool_calling_agent(research_llm, research_tools, research_prompt)
research_executor = AgentExecutor(
    agent=research_agent,
    tools=research_tools,
    verbose=True,
    max_iterations=5
)

print("Research agent running...\n")
research_response = research_executor.invoke({
    "input": (
        "Search for information about the Transformer neural network architecture. "
        "Also calculate statistics for these model accuracy scores: 91.5, 93.2, 92.8, 94.1, 90.7, 93.6"
    )
})

print("\nAgent final response:")
print(research_response["output"])

---
## Use Case 3: Customer Support Bot with Persistent Memory

**Problem:** A SaaS startup wants an automated support assistant that maintains context throughout a customer's session. Without memory, the bot treats every message as a fresh conversation, forcing users to repeat themselves constantly.

**Solution:** A support bot built with LangChain's memory system. The bot maintains a complete conversation history per session and uses a specialized system prompt that defines its persona and communication style. Users only need to state their name and issue once.

**Components used:** ChatGoogleGenerativeAI, RunnableWithMessageHistory, InMemoryChatMessageHistory, ChatPromptTemplate, StrOutputParser

In [ ]:
# Use Case 3: Stateful customer support bot

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# System prompt defines the bot's persona and behavioral guidelines
support_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are Aria, a customer support specialist for TechFlow, a cloud-based project management platform.

Guidelines:
- Address the user by name whenever possible
- Keep responses focused and solution-oriented
- Acknowledge the user's frustration when appropriate
- If you cannot resolve an issue, escalate clearly
- Maintain context from earlier in the conversation at all times"""),
    ("placeholder", "{chat_history}"),
    ("human", "{input}")
])

# Session storage
support_store = {}

def get_support_session(session_id: str):
    """Retrieve or initialize a conversation history for this support ticket."""
    if session_id not in support_store:
        support_store[session_id] = InMemoryChatMessageHistory()
    return support_store[session_id]

support_llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0.3,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

support_base_chain = support_prompt | support_llm | StrOutputParser()

support_bot = RunnableWithMessageHistory(
    support_base_chain,
    get_support_session,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Simulate a multi-turn support conversation
ticket_config = {"configurable": {"session_id": "ticket_4821"}}

conversation_turns = [
    "Hi, my name is Priya. I am unable to log into my account since this morning.",
    "I already tried resetting my password twice but it is still not working.",
    "Yes I am using the mobile app. Also, do you know what my name is and what the issue is?"
]

print("Support conversation simulation")
print("=" * 50)

for message in conversation_turns:
    print(f"\nUser: {message}")
    bot_reply = support_bot.invoke({"input": message}, config=ticket_config)
    print(f"Aria: {bot_reply}")
    print("-" * 40)

---
## Summary

This notebook demonstrated the complete LangChain toolkit from individual components through to production-grade application patterns.

| Component | Purpose |
|---|---|
| ChatGoogleGenerativeAI | Gemini model interface via LangChain abstraction |
| ChatPromptTemplate | Reusable, parameterized prompt structures |
| LCEL (pipe operator) | Composing components into readable pipelines |
| RunnableWithMessageHistory | Session-aware conversation memory |
| @tool decorator | Wrapping functions for agent use |
| AgentExecutor | Autonomous reasoning loop with tool selection |
| WebBaseLoader | Loading and parsing web documents |
| FAISS | Local vector storage for semantic retrieval |
| RAG chain | Grounding LLM answers in retrieved facts |

The key insight across all of this is that LangChain's value is not any single component in isolation. It is the standardized interface that lets all these components plug into each other, and the ability to swap out the underlying model or data source without touching the rest of your application.

The next step beyond what is covered here is LangGraph, which extends LangChain's model to support graph-based multi-agent workflows where multiple specialized agents collaborate on complex tasks that require branching logic and iterative refinement.